# Useful References: Importance Sampling, PPO and GRPO

## Intro

Training modern LLMs with reinforcement learning (RLHF, RLVR, RL on verifiable rewards) rests on a
small set of core ideas: **importance sampling**, the **clipped PPO objective**, and the **group-relative**
advantage normalization popularized by **GRPO**.

This page distills Nando de Freitas' (@NandoDF) compact handwritten notes on exactly that stack —
from the definition of the RL objective all the way to the full GRPO-style objective with the
unbiased k3 KL estimator. It is a dense but self-contained refresher: notation first, then the
estimators, then the algorithms.

- X post: <https://x.com/NandoDF/status/1919728246821634205>
- Attached X Article: <https://x.com/i/article/1919697329487069184> ("May 6: Importance sampling, PPO and GRPO")

## Notation

- $o$ — observation (prompt)
- $a$ — action (completion)
- $\theta$ — policy parameters
- $R = r(a, o)$ — the reward
- $D = \{o^i, a^i, r(a^i, o^i)\}_{i=1}^N$ — dataset of $N$ samples

## The objective

$$J(\theta) = E[r(a, o)] = \int r(a, o)\, \pi_\theta(a|o)\, P(o)\, da\, do = E_{\pi_\theta}[r(a, o)]$$

## Importance sampling

Sample from a different (old/behavior) policy $\mu$ instead of $\pi_\theta$:

$$w(\theta) = \frac{\pi_\theta(a|o)}{\mu(a|o)}$$
$$J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} w(\theta^i)\, r(a^i, o^i), \quad (o^i, a^i) \sim \mu$$

Per-sample importance weights against the old policy:

$$w^i = \frac{\pi_\theta(a^i|o^i)}{\pi_{\mathrm{old}}(a^i|o^i)}$$
$$J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} w^i\, r(a^i, o^i)$$

### Variance of the weights

$$\mathrm{Var}[w] = E[w^2] - (E[w])^2, \qquad E[w] = 1 \quad \left(\text{since } \int \pi_\theta(a|o)\, da = 1\right)$$
$$E[w^2] = E_{\pi_{\mathrm{old}}}\left[\left(\frac{\pi_\theta}{\pi_{\mathrm{old}}}\right)^2\right] = E_{\pi_\theta}\left[\frac{\pi_\theta}{\pi_{\mathrm{old}}}\right]$$

The variance blows up when $\pi_\theta / \pi_{\mathrm{old}}$ is large — i.e. when the policies differ a lot.

### Self-normalized weights

Trades a little bias for lower variance:

$$\tilde{w}^i = \frac{w^i}{\sum_j w^j}, \qquad \text{so that } \sum_i \tilde{w}^i = 1$$

## Baselines and advantage

A baseline $b$ that is independent of $a^i$ does not change the expectation:

$$A(a, o) = r(a, o) - b$$
$$J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} w^i\, (r^i - b)$$

**RLOO** (REINFORCE leave-one-out baseline): the baseline is the mean reward of the *other*
samples for the same prompt:

$$b = \frac{1}{N-1} \sum_{j \ne i} r(a^j, o^j)$$
$$A^i = r(a^i, o^i) - \frac{1}{N-1} \sum_{j \ne i} r(a^j, o^j)$$

## PPO (clipped surrogate objective — pessimistic update)

$$J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} \min\big( w^i A^i,\; \mathrm{clip}(w^i, 1-\varepsilon, 1+\varepsilon)\, A^i \big)$$

Clipping: when $A^i > 0$ and $w^i > 1+\varepsilon$ the update is clipped (don't push the policy too
far for good actions); likewise when $A^i < 0$ and $w^i < 1-\varepsilon$. This limits how much the
new policy can deviate from the old one.

## KL penalty: the k3 estimator

Unbiased and always positive. Per-sample KL term against the SFT reference policy:

$$-\log\left(\frac{\pi_{\mathrm{SFT}}(a^i|o^i)}{\pi_\theta(a^i|o^i)}\right) + \frac{\pi_{\mathrm{SFT}}(a^i|o^i)}{\pi_\theta(a^i|o^i)} - 1$$

## GRPO (group-relative advantage)

Normalize rewards within the group of samples for a prompt:

$$A^i = \frac{r^i - \mathrm{mean}(r)}{\mathrm{std}(r)}$$

Full GRPO-style objective (PPO clipping + KL penalty against the SFT reference policy):

$$J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} \min\left( \frac{\pi_\theta(a^i|o^i)}{\pi_{\mathrm{old}}(a^i|o^i)} r^i,\; \mathrm{clip}\left[\frac{\pi_\theta(a^i|o^i)}{\pi_{\mathrm{old}}(a^i|o^i)}, 1-\varepsilon, 1+\varepsilon\right] r^i \right)$$
$$- \frac{\beta}{N} \sum_{i=1}^{N} \left[ -\log\left(\frac{\pi_{\mathrm{SFT}}(a^i|o^i)}{\pi_\theta(a^i|o^i)}\right) + \frac{\pi_{\mathrm{SFT}}(a^i|o^i)}{\pi_\theta(a^i|o^i)} - 1 \right]$$

## Further reading (mentioned in replies to the post)

- <https://arxiv.org/html/2503.1428>